In [2]:
from parsing_library import *

# Repeated experiments

In [7]:
## ---- from saved calculate_coverage.py json files -----

import os
import json

saved_coverage_dir = "../../coverage_ten_V100_K10_VI"

# Dictionary to store sums and counts for averaging
size_sums = {}
#size_counts = {}
count = 0
# Traverse through subdirectories and collect JSON files
for root, dirs, files in os.walk(saved_coverage_dir):
    for file in files:
        if file.endswith(".json"):
            json_path = os.path.join(root, file)
            print(json_path)
            with open(json_path, 'r') as f:
                data = json.load(f)

            # Sum values for each size
            for size, value in data.items():
                size = int(size) 
                if size not in size_sums:
                    size_sums[size] = 0
                    if size_sums[size] == 0 and count > 0:
                        print(f'missing size? {json_path}, {size}')
                size_sums[size] += value
            count += 1

averages = [f"{(size_sums[size] / count):.1f}" for size in sorted(size_sums.keys())]
result_string = " & ".join(averages)

print(result_string)


../../coverage_ten_V100_K10_VI/coverage_1.json
../../coverage_ten_V100_K10_VI/coverage_5.json
../../coverage_ten_V100_K10_VI/coverage_7.json
../../coverage_ten_V100_K10_VI/coverage_4.json
../../coverage_ten_V100_K10_VI/coverage_3.json
../../coverage_ten_V100_K10_VI/coverage_10.json
../../coverage_ten_V100_K10_VI/coverage_6.json
../../coverage_ten_V100_K10_VI/coverage_2.json
../../coverage_ten_V100_K10_VI/coverage_8.json
../../coverage_ten_V100_K10_VI/coverage_9.json
92.5 & 91.9 & 90.4 & 88.0 & 83.7 & 85.4 & 85.6 & 85.1 & 83.0 & 78.9


In [2]:
from parsing_library import *

# Assume:
# - ten_2d_datasets contains datasets, ex: ten_2d_datasets/1.json
# - ten_2d_datasets/inference_method/1 are the fit folders

main_folder = '../../data/ten_V100K5-zipf'#'ten_2d_datasets' 
#saved_fit_folder = 'ten_V100K5/nofix/vi/'
saved_fit_folder = '../../results_ten_V100K5_zipf/nofix/vi'

data_folder = '../../data/ten_V100K5-zipf'
inference_method = 'vi'

fit_folder_base_name = "%i-zipf"
# fit_folder_base_name = %i"
base_data_name = "%i-zipf.json"
#base_data_name = "%i.json"


experiments_ids = [1]#,2,3, 4, 5, ,6, 7, 8, 9, 10]
#sizes = [100, 200, 500, 1000, 5000, 20000, 50000, 100000]
sizes = [1000]#, 2000, 5000, 10000, 20000, 50000, 100000]

coverage_per_size = {size: [] for size in sizes}

for size in sizes:
    print(f"Processing size {size}...")
    
    for repetition in experiments_ids:
        covered_count = 0
        
        # data
        data_path = os.path.join(data_folder,base_data_name%repetition)
        with open(data_path) as f:
            data = json.load(f)

        true_theta = np.array(data['theta'])
        vocabulary = data['vocabulary']
        word_pairs = list(itertools.combinations(vocabulary, 2))
        total_pairs = len(word_pairs) * 2

        D = true_theta.shape[1]
        print('dim found from true, D = ', D)

        # inference fit path
        saved_fit_dir_i = os.path.join(saved_fit_folder, fit_folder_base_name%repetition)
        
        if not inference_method == 'gibbs':
            fit = load_fit_by_size(size, saved_fit_dir_i) #load_fit

        if inference_method == 'hmc':
            estimated_word_vectors = fit['word_vectors']
            estimated_context_vectors = fit['context_vectors']
            num_samples = estimated_word_vectors.shape[2]
        elif inference_method == 'vi':
            estimated_word_vectors, estimated_context_vectors = extract_word_and_context_vectors_vi(fit.variational_sample_pd, vocabulary, D=D)
            num_samples = estimated_word_vectors.shape[2]
        elif inference_method in ('laplace', 'map'):
            # use same extractor as for VI.
            estimated_word_vectors, estimated_context_vectors = extract_word_and_context_vectors_vi(fit.draws_pd(), vocabulary)
            num_samples = estimated_word_vectors.shape[2]
        elif inference_method == 'gibbs':
            filepath = os.path.join(saved_fit_dir_i, f'gibbs-samples-N-{size}.csv.fix')
            estimated_word_vectors, estimated_context_vectors = extract_word_and_context_vectors_gibbs(filepath, vocabulary)
            num_samples = estimated_word_vectors.shape[2]

        theta_samples = []
        for i in range(num_samples):
            word_vectors = estimated_word_vectors[:, :, i]
            context_vectors = estimated_context_vectors[:, :, i]
            theta = np.vstack((word_vectors, context_vectors))
            theta_samples.append(theta)

        # loop through word pairs and compute coverage
        for word1, word2 in word_pairs:
            vi = vocabulary[word1]
            wi = vocabulary[word2]
            true_p = calculate_p(vi, wi, true_theta, len(vocabulary))

            eta_samples = [calculate_p(vi, wi, theta, len(vocabulary)) for theta in theta_samples]
            lower_ci, upper_ci = credible_interval(eta_samples)

            if lower_ci <= true_p <= upper_ci:
                covered_count += 1

            true_p = calculate_p(wi, vi, true_theta, len(vocabulary))

            eta_samples = [calculate_p(wi, vi, theta, len(vocabulary)) for theta in theta_samples]
            lower_ci, upper_ci = credible_interval(eta_samples)

            if lower_ci <= true_p <= upper_ci:
                covered_count += 1

        coverage_percentage = (covered_count / total_pairs) * 100
        coverage_per_size[size].append(coverage_percentage)

# Average the coverage for each size
average_coverage_per_size = {size: np.mean(coverage_per_size[size]) for size in sizes}


coverage_stats_per_size = {}
for size in sizes:
    coverage_list = coverage_per_size[size]
    mean_coverage = np.mean(coverage_list)
    std_coverage = np.std(coverage_list)  # Standard deviation
    sem_coverage = std_coverage / np.sqrt(len(coverage_list))  # SEM
    coverage_stats_per_size[size] = {'mean': mean_coverage, 'std': std_coverage, 'sem': sem_coverage}


for size, stats in coverage_stats_per_size.items():
    print(f"Data size: {size}, Average coverage: {stats['mean']:.2f}%, ± {stats['std']:.2f}% (std), ± {stats['sem']:.2f}% (SEM)")



KeyboardInterrupt: 

In [7]:
true_theta.shape

(200, 5)

,"word_vectors[1,1]","word_vectors[2,1]","word_vectors[3,1]","word_vectors[4,1]","word_vectors[5,1]","word_vectors[6,1]","word_vectors[7,1]","word_vectors[8,1]","word_vectors[9,1]","word_vectors[10,1]",...,"context_vectors[91,5]","context_vectors[92,5]","context_vectors[93,5]","context_vectors[94,5]","context_vectors[95,5]","context_vectors[96,5]","context_vectors[97,5]","context_vectors[98,5]","context_vectors[99,5]","context_vectors[100,5]"
0,-0.181118,0.063709,0.807181,0.209123,-0.465466,-0.312532,0.396297,-0.194098,0.182329,-0.075630,...,0.630300,-0.818340,0.062275,-0.670944,0.234504,0.059875,-0.442231,-0.411227,-0.278312,-0.183501
1,-0.452790,0.099036,-0.494325,0.012580,0.466670,-0.210629,0.007283,0.592905,1.293100,0.026335,...,-0.175447,0.007890,0.419220,0.952163,0.317098,-0.628969,-0.163730,0.231910,0.214442,-0.207430
2,0.085269,-0.132723,-0.082014,-0.428403,-0.208334,-0.907142,-0.360229,-0.409505,-0.683609,0.282377,...,0.061104,-0.545898,0.328712,-0.288337,0.010952,-0.351312,0.271237,-0.388865,0.554200,-0.073337
3,0.117802,0.044739,-0.490126,0.132755,-0.330051,-0.196334,-0.815041,0.176397,-0.221915,-0.513476,...,-0.699064,0.282789,-0.348762,-0.319417,0.137298,-0.385938,-0.291200,-0.063809,0.477972,0.554691
4,0.420045,-0.132809,-0.127364,-0.251751,0.037494,-0.194692,-0.215982,0.652954,0.912565,0.111145,...,-0.200333,-1.033380,-0.588907,0.074064,0.504864,-0.104331,0.074586,-0.973243,0.613461,-0.124497
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,0.064106,-0.244252,0.314351,-0.229577,0.190953,0.106050,0.612887,0.003873,0.018446,-0.419479,...,0.396948,-0.534057,-1.712240,0.283012,-0.199916,0.343185,0.441073,0.092064,1.061370,0.432827
1996,0.040263,0.320885,0.571008,1.113290,-0.576109,-0.008205,-0.240624,0.206562,0.116767,-0.433599,...,0.346155,0.002405,-0.385960,0.346343,-1.113850,-0.802516,-0.020392,-0.003806,0.502980,0.726556
1997,-0.091418,0.631042,0.030248,0.066079,-0.163822,0.453876,0.743977,0.103411,-0.131909,0.509521,...,-0.410795,-0.072789,0.061595,-0.572969,0.189919,0.127367,-0.287921,0.771024,0.352035,-0.409704
1998,0.401089,-0.227088,-0.382101,0.248019,0.007348,-0.060785,-0.290244,-0.547807,-0.099688,0.152756,...,0.626078,-0.012729,0.355407,0.536165,0.210123,0.269279,0.183292,-0.151652,0.042935,-0.314860
